# Demo 03 : Human-in-the-loop middleware

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · approval-policy

We gate a simulated part reservation with `HumanInTheLoopMiddleware`. We use fresh threads for approval, editing and rejection. The model chooses the proposed arguments, so we inspect the pending action before supplying a decision.

In [2]:
import json
import operator
import sqlite3
import time
from pathlib import Path
from typing import Annotated, Literal, TypedDict
from uuid import uuid4

from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain.tools import tool
from langchain_core.callbacks import BaseCallbackHandler
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Command, interrupt

class Usage(BaseCallbackHandler):
    def __init__(self):
        self.calls = self.input_tokens = self.output_tokens = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        self.calls += 1

    def on_llm_end(self, response, **kwargs):
        for batch in response.generations:
            for generation in batch:
                usage = generation.message.usage_metadata or {}
                self.input_tokens += usage.get("input_tokens", 0)
                self.output_tokens += usage.get("output_tokens", 0)

usage = Usage()
llm = make_llm(callbacks=[usage], max_retries=0, max_tokens=650)
started = time.perf_counter()
DB = Path(data("demo-world/service.db")).resolve()
with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as conn:
    conn.row_factory = sqlite3.Row
    work_order = dict(conn.execute("SELECT * FROM work_orders WHERE wo_id = ?", ("BW-7001",)).fetchone())
    part = dict(conn.execute("SELECT * FROM parts WHERE part_no = ?", ("CE-CF-02",)).fetchone())
print("work order:", work_order)
print("part:", part)

def config(label):
    return {"configurable": {"thread_id": f"07-{label}-{uuid4()}"}, "recursion_limit": 30}

work order: {'wo_id': 'BW-7001', 'fleet_no': 'CE12-2021-0117', 'opened': '2026-09-14', 'status': 'open', 'error_code': 'F-CE-110', 'summary': 'Traction system over-temperature during afternoon route', 'mechanic_id': 'M01'}
part: {'part_no': 'CE-CF-02', 'model': 'CE-12', 'name': 'traction cooling fan', 'stock': 6, 'price_eur': 420.0, 'depot': 'Rietpoort'}


In [3]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

reservations = []

@tool
def reserve_part(part_no: str, quantity: int) -> str:
    """Record a simulated reservation of a Veenhaven Bus part, without updating the database."""
    if part_no != part["part_no"] or not 0 < quantity <= part["stock"]:
        raise ValueError("Invalid reservation")
    reservations.append({"part_no": part_no, "quantity": quantity})
    return json.dumps(reservations[-1])

policy = HumanInTheLoopMiddleware(interrupt_on={
    "reserve_part": {"allowed_decisions": ["approve", "edit", "reject"]}
})
agent = create_agent(llm, tools=[reserve_part], middleware=[policy],
    system_prompt="Call reserve_part once with the requested arguments. After a tool result, report it and stop. After rejection, stop without retrying. An edited result is authoritative.",
    checkpointer=InMemorySaver())

for choice in ("approve", "edit", "reject"):
    cfg = config(choice)
    before = len(reservations)
    out = agent.invoke({"messages": [("human", f"Reserve 2 of {part['part_no']} for BW-7001.")]}, cfg, version="v2")
    assert len(out.interrupts) == 1 and len(reservations) == before
    pending = out.interrupts[0].value
    print("actions:", pending["action_requests"])
    print("allowed:", pending["review_configs"])
    assert len(pending["action_requests"]) == 1
    decision = {"type": choice}
    if choice == "edit":
        decision["edited_action"] = {"name": "reserve_part", "args": {"part_no": part["part_no"], "quantity": 1}}
    if choice == "reject":
        decision["message"] = "Reservation denied. Abandon the reservation and do not retry."
    out = agent.invoke(Command(resume={"decisions": [decision]}), cfg, version="v2")
    assert not out.interrupts
    print("decision:", choice)
    print("new reservations:", reservations[before:])
    for message in out.value["messages"]:
        if isinstance(message, AIMessage) and message.tool_calls:
            print("model call:", message.tool_calls)
        if isinstance(message, ToolMessage):
            print("tool:", message.status, message.content)
    print("answer:", out.value["messages"][-1].content)
    assert len(reservations) - before == (0 if choice == "reject" else 1)
    if choice == "edit":
        assert reservations[-1]["quantity"] == 1

actions: [{'name': 'reserve_part', 'args': {'part_no': 'CE-CF-02', 'quantity': 2}, 'description': "Tool execution requires approval\n\nTool: reserve_part\nArgs: {'part_no': 'CE-CF-02', 'quantity': 2}"}]
allowed: [{'action_name': 'reserve_part', 'allowed_decisions': ['approve', 'edit', 'reject']}]


decision: approve
new reservations: [{'part_no': 'CE-CF-02', 'quantity': 2}]
model call: [{'name': 'reserve_part', 'args': {'part_no': 'CE-CF-02', 'quantity': 2}, 'id': 'call_jlo97TrgDmClWNr2q9iIlM2M', 'type': 'tool_call'}]
tool: success {"part_no": "CE-CF-02", "quantity": 2}
answer: Reserved 2 of CE-CF-02 for BW-7001.


actions: [{'name': 'reserve_part', 'args': {'part_no': 'CE-CF-02', 'quantity': 2}, 'description': "Tool execution requires approval\n\nTool: reserve_part\nArgs: {'part_no': 'CE-CF-02', 'quantity': 2}"}]
allowed: [{'action_name': 'reserve_part', 'allowed_decisions': ['approve', 'edit', 'reject']}]


decision: edit
new reservations: [{'part_no': 'CE-CF-02', 'quantity': 1}]
model call: [{'name': 'reserve_part', 'args': {'part_no': 'CE-CF-02', 'quantity': 2}, 'id': 'call_K2RsyHZCUqTpKqvDYQlfOhgl', 'type': 'tool_call'}]
tool: success Note: a human reviewer replaced this tool call before it ran. The call recorded in your message is the one you produced, not the one that executed. This was intentional and authorized. Do not re-issue your original call. Executed instead: reserve_part with arguments {"part_no": "CE-CF-02", "quantity": 1}.

Tool response:
{"part_no": "CE-CF-02", "quantity": 1}
answer: Reserved 1 of CE-CF-02.


actions: [{'name': 'reserve_part', 'args': {'part_no': 'CE-CF-02', 'quantity': 2}, 'description': "Tool execution requires approval\n\nTool: reserve_part\nArgs: {'part_no': 'CE-CF-02', 'quantity': 2}"}]
allowed: [{'action_name': 'reserve_part', 'allowed_decisions': ['approve', 'edit', 'reject']}]


decision: reject
new reservations: []
model call: [{'name': 'reserve_part', 'args': {'part_no': 'CE-CF-02', 'quantity': 2}, 'id': 'call_iZSuh1mek4cDUKfAJ79HfFM6', 'type': 'tool_call'}]
tool: error User rejected the tool call for `reserve_part` with reason: Reservation denied. Abandon the reservation and do not retry.
answer: The reservation for 2 of part CE-CF-02 was denied.


We return one decision per pending action, in the action list's order. Editing uses `edited_action.args`, the same key as the request's `args`. On the installed stack the AI message keeps its original call; the tool result includes a reviewer-edit notice. Rejection creates an error tool message without executing the tool.

## b · per-call-approval

We compare the batched form with `interrupt_mode="per_call"`. A scripted model proposes the same two reservations in one turn to both agents, so only the mode differs. The door actuator is out of stock, and the reviewer answers that call with `respond` instead of running the tool.

In [4]:
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from pydantic import ValidationError

STOCK = {"CE-CF-02": part["stock"], "CE-DA-03": 0}

@tool
def reserve_stock(part_no: str, quantity: int) -> str:
    """Record a simulated reservation of a Veenhaven Bus part, without updating the database."""
    if not 0 < quantity <= STOCK.get(part_no, 0):
        raise ValueError("Invalid reservation")
    reservations.append({"part_no": part_no, "quantity": quantity})
    return json.dumps(reservations[-1])

class Scripted(GenericFakeChatModel):
    def bind_tools(self, tools, **kwargs):
        return self

def gated_agent(mode):
    script = Scripted(messages=iter([
        AIMessage("", tool_calls=[
            {"name": "reserve_stock", "args": {"part_no": "CE-CF-02", "quantity": 2}, "id": "call-fan"},
            {"name": "reserve_stock", "args": {"part_no": "CE-DA-03", "quantity": 1}, "id": "call-door"}]),
        AIMessage("Reservations handled.")]))
    gate = HumanInTheLoopMiddleware(interrupt_on={"reserve_stock": True}, interrupt_mode=mode)
    return create_agent(script, tools=[reserve_stock], middleware=[gate], checkpointer=InMemorySaver())

request = {"messages": [HumanMessage("Reserve the fan and the door actuator for BW-7001.")]}
door_reply = {"type": "respond", "message": "CE-DA-03 is out of stock ; the depot reorders it."}

reservations.clear()
batched = gated_agent("batched")
batch_cfg = config("batched")
out = batched.invoke(request, batch_cfg, version="v2")
print("batched interrupts:", len(out.interrupts), [a["args"] for a in out.interrupts[0].value["action_requests"]])
out = batched.invoke(Command(resume={"decisions": [{"type": "approve"}, door_reply]}), batch_cfg, version="v2")
print("batched results:", [m.content for m in out.value["messages"] if isinstance(m, ToolMessage)])
print("batched reservations:", reservations)

reservations.clear()
per_call = gated_agent("per_call")
call_cfg = config("per-call")
out = per_call.invoke(request, call_cfg, version="v2")
pending = {item.value["args"]["part_no"]: item for item in out.interrupts}
print("per-call interrupts:", len(out.interrupts))
print("door request:", pending["CE-DA-03"].value)
try:
    per_call.invoke(Command(resume={pending["CE-DA-03"].id: {"type": "edit", "edited_action": {"name": "cancel_order", "args": {}}}}), call_cfg, version="v2")
except ValidationError as exc:
    print("invalid answer:", type(exc).__name__, "; still pending:", len(per_call.get_state(call_cfg).interrupts))
out = per_call.invoke(Command(resume={pending["CE-CF-02"].id: {"type": "approve"}}), call_cfg, version="v2")
print("after the fan answer:", len(out.interrupts), "pending ; reservations:", reservations)
out = per_call.invoke(Command(resume={pending["CE-DA-03"].id: door_reply}), call_cfg, version="v2")
print("per-call results:", [m.content for m in out.value["messages"] if isinstance(m, ToolMessage)])
assert not out.interrupts and reservations == [{"part_no": "CE-CF-02", "quantity": 2}]

batched interrupts: 1 [{'part_no': 'CE-CF-02', 'quantity': 2}, {'part_no': 'CE-DA-03', 'quantity': 1}]
batched results: ['CE-DA-03 is out of stock ; the depot reorders it.', '{"part_no": "CE-CF-02", "quantity": 2}']
batched reservations: [{'part_no': 'CE-CF-02', 'quantity': 2}]
per-call interrupts: 2
door request: {'type': 'tool_approval', 'tool_call_id': 'call-door', 'name': 'reserve_stock', 'args': {'part_no': 'CE-DA-03', 'quantity': 1}, 'description': "Tool execution requires approval\n\nTool: reserve_stock\nArgs: {'part_no': 'CE-DA-03', 'quantity': 1}"}
invalid answer: ValidationError ; still pending: 2
after the fan answer: 1 pending ; reservations: [{'part_no': 'CE-CF-02', 'quantity': 2}]
per-call results: ['{"part_no": "CE-CF-02", "quantity": 2}', 'CE-DA-03 is out of stock ; the depot reorders it.']


Batched mode asks once per model turn and runs nothing until every decision arrives. Per-call mode raises one interrupt per gated call, keyed by its id, and checks each answer against a schema before saving it. The edit that tried to switch tools was refused and left the call pending. The approved fan ran while the door call still waited. LangChain recommends per-call mode for new agents.

In [5]:
print("metrics:", json.dumps({"model": MODEL, "calls": usage.calls,
      "input_tokens": usage.input_tokens, "output_tokens": usage.output_tokens,
      "seconds": round(time.perf_counter() - started, 1)}))

metrics: {"model": "openai/gpt-6-luna", "calls": 6, "input_tokens": 908, "output_tokens": 133, "seconds": 16.6}


## Things to try

Optional : we can restrict the policy to approve and reject, or request several reservations and supply one decision per action.